# Energy Market Data Exploration

This notebook is the first exploratory consumer of `YFinanceDailyAdapter`. It pulls daily Yahoo Finance series into the core `DataService`, then builds a few lightweight views for the energy/oil story.

The futures-like Yahoo symbols here (`CL=F`, `BZ=F`, and related fuels) are continuous front-month-style proxies. They are useful for orientation, but not a substitute for a contract-level futures data source.

In [ ]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data.py").exists():
    NOTEBOOK_DIR = Path("playground/energy_yfinance").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from data import CATEGORY_LABELS, build_energy_market_service  # noqa: E402


pd.options.display.float_format = "{:.3f}".format
plt.style.use("seaborn-v0_8-whitegrid")

## Load Market Data

The first run downloads from Yahoo Finance and writes parquet files under the repo-root `data/yfinance/` cache. Warm-cache runs do not touch the network unless `REFRESH = True`.

In [ ]:
def find_repo_root(start: Path) -> Path:
    """Walk upward until the repository root pyproject is found."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "aieng-forecasting").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CACHE_DIR = REPO_ROOT / "data" / "yfinance"
START = "2005-01-01"
END = None
REFRESH = False

service = build_energy_market_service(start=START, end=END, cache_dir=CACHE_DIR, refresh=REFRESH)
summary = service.summary()
summary.insert(1, "label", summary["series_id"].map(CATEGORY_LABELS))
summary

In [ ]:
as_of = datetime.now()
frames: list[pd.DataFrame] = []
for series_id in service.series_ids:
    frame = service.get_series(series_id, as_of=as_of)[["timestamp", "value"]].copy()
    frame["series_id"] = series_id
    frames.append(frame)

prices = pd.concat(frames, ignore_index=True)
wide = prices.pivot(index="timestamp", columns="series_id", values="value").sort_index()
wide.tail()

## Normalized Price Levels

Each series starts at 100 from its first available observation, making long-run co-movement easier to compare despite different units and start dates.

In [ ]:
def normalize_to_first_observation(series: pd.Series) -> pd.Series:
    """Scale a price series to 100 at its first valid observation."""
    valid = series.dropna()
    if valid.empty:
        return series
    return series / valid.iloc[0] * 100


normalized = wide.apply(normalize_to_first_observation).rename(columns=CATEGORY_LABELS)
ax = normalized.plot(figsize=(12, 6), linewidth=1.4)
ax.set_title("Energy and Related Market Series, Normalized To First Observation")
ax.set_ylabel("Index, first observation = 100")
ax.set_xlabel("")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()

## Recent Window

The recent window is more useful for presentation framing because the May 2026 story is about current conditions and forward-looking risks.

In [ ]:
recent_start = wide.index.max() - pd.DateOffset(years=2)
recent = wide.loc[wide.index >= recent_start]
recent_normalized = recent.apply(normalize_to_first_observation).rename(columns=CATEGORY_LABELS)

ax = recent_normalized.plot(figsize=(12, 6), linewidth=1.6)
ax.set_title("Recent Energy and Market Context")
ax.set_ylabel("Index, recent window start = 100")
ax.set_xlabel("")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()

## Returns, Volatility, And Correlation

Daily returns make cross-market relationships easier to inspect than raw levels. This is still descriptive, not a forecasting model.

In [ ]:
returns = wide.pct_change(fill_method=None).dropna(how="all")
annualized_vol = returns.rolling(63).std() * (252**0.5)

oil_and_fuels = [
    "wti_crude_oil_front_month",
    "brent_crude_oil_front_month",
    "rbob_gasoline_front_month",
    "heating_oil_front_month",
    "natural_gas_front_month",
]
ax = annualized_vol[oil_and_fuels].rename(columns=CATEGORY_LABELS).plot(figsize=(12, 5), linewidth=1.3)
ax.set_title("Rolling 63-Trading-Day Annualized Volatility")
ax.set_ylabel("Annualized volatility")
ax.set_xlabel("")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()

In [ ]:
recent_returns = returns.loc[returns.index >= returns.index.max() - pd.DateOffset(years=3)]
correlation = recent_returns.rename(columns=CATEGORY_LABELS).corr()
correlation.style.format("{:.2f}").background_gradient(cmap="coolwarm", vmin=-1, vmax=1)

## Fast Follow: Futures Data

Before treating futures seriously, investigate contract-level sources, roll rules, curve snapshots, open interest, volume, and licensing. Yahoo Finance is enough for this first exploratory pass, but not enough for futures-curve semantics.